# 🚀 Hunyuan3D-2.1 Cloud Server for Unity Editor Bridge

Run the official **Hunyuan3D-2.1** inference server on a **free Google Colab GPU (T4 / A100 / L4)** and connect it directly to your Unity Editor project.

### 📌 How to use:
1. In the Colab menu, go to **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** (free) or better.
2. Run all cells below (or press **Ctrl + F9**).
3. Wait for the final cell to display your **Public Cloudflare Tunnel URL** (e.g. `https://xxxx.trycloudflare.com`).
4. Copy and paste that URL into the **Server URL** field inside Unity (**Tools ➔ Hunyuan3D ➔ Generator**).
5. Click **Check Server** in Unity ➔ Ready to generate 3D models from any Mac, laptop, or low-end PC!

In [ ]:
# Step 1: Check GPU and enable 12GB Swap Memory (prevents Colab RAM crashes)
!nvidia-smi
# Create 12GB swap to expand Colab System RAM from 12GB to 24GB+
!fallocate -l 12G /swapfile 2>/dev/null || dd if=/dev/zero of=/swapfile bs=1M count=12288
!chmod 600 /swapfile
!mkswap /swapfile
!swapon /swapfile 2>/dev/null || true
!free -h


In [ ]:
# Step 2: Clone official Tencent Hunyuan3D-2 repository
import os
if not os.path.exists('Hunyuan3D-2'):
    !git clone https://github.com/Tencent/Hunyuan3D-2.1.git Hunyuan3D-2 || git clone https://github.com/Tencent/Hunyuan3D-2.git Hunyuan3D-2
%cd Hunyuan3D-2

In [ ]:
# Step 3: Install dependencies for shape generation
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install fastapi uvicorn pydantic trimesh rembg
# Install cloudflared to create a free public HTTPS tunnel without account/token
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# Step 4: Apply Unity Bridge optimization patch (step-limit, shape-only, VRAM cleanup)
import os, re

if os.path.exists('api_models.py'):
    with open('api_models.py', 'r') as f: content = f.read()
    content = re.sub(r'le=20', 'le=50', content)
    with open('api_models.py', 'w') as f: f.write(content)
    print('Patched api_models.py (inference steps allowed up to 50)')

if os.path.exists('model_worker.py'):
    with open('model_worker.py', 'r') as f: content = f.read()
    modified = False
    # wrap texture pipeline in try-except for shape-only stability
    if 'except Exception as e:' not in content and 'self.paint_pipeline = Hunyuan3DPaintPipeline(conf)' in content:
        content = content.replace('self.paint_pipeline = Hunyuan3DPaintPipeline(conf)',
            'try:\n            self.paint_pipeline = Hunyuan3DPaintPipeline(conf)\n        except:\n            self.paint_pipeline = None')
        modified = True
    # Inject VRAM cleanup helper after imports
    if '# [Unity Bridge] VRAM cleanup' not in content:
        vram_code = '\n# [Unity Bridge] VRAM cleanup\nimport gc as _gc\ndef _cleanup_vram():\n    _gc.collect()\n    try:\n        import torch\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n    except: pass\n'
        m = re.search(r'^(class\s+\w+)', content, re.MULTILINE)
        if m:
            content = content[:m.start()] + vram_code + '\n' + content[m.start():]
            modified = True
            print('Injected _cleanup_vram() into model_worker.py')
    if modified:
        with open('model_worker.py', 'w') as f: f.write(content)
    print('Patched model_worker.py')

# Inject POST /unload endpoint for on-demand VRAM release
if os.path.exists('api_server.py'):
    with open('api_server.py', 'r') as f: content = f.read()
    if '# [Unity Bridge] VRAM unload' not in content:
        ep = ('\n# [Unity Bridge] Healthcheck & Root endpoints\n'
      '@app.get("/")\n'
      'async def root():\n'
      '    return {"status": "healthy", "message": "Hunyuan3D-2.1 Server is running"}\n\n'
      '@app.get("/health")\n'
      'async def health():\n'
      '    return {"status": "healthy", "worker_id": "hunyuan-worker-1"}\n\n'
      '# [Unity Bridge] VRAM unload endpoint\n'
              '@app.post("/unload")\n'
              'async def unload_vram():\n'
              '    import gc, torch\n'
              '    freed = 0\n'
              '    try:\n'
              '        if torch.cuda.is_available():\n'
              '            before = torch.cuda.memory_allocated()\n'
              '            gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()\n'
              '            freed = (before - torch.cuda.memory_allocated()) / 1024 / 1024\n'
              '    except Exception as e:\n'
              '        return {"status": "error", "message": str(e)}\n'
              '    return {"status": "ok", "freed_mb": round(freed, 1)}\n')
        if 'if __name__' in content:
            content = content.replace('if __name__', ep + '\nif __name__')
        else:
            content += ep
        with open('api_server.py', 'w') as f: f.write(content)


In [ ]:
# Step 5: Start Hunyuan3D FastAPI server and Cloudflare Tunnel
import os, subprocess, time, re, urllib.request

# Optimize PyTorch CUDA memory management to prevent VRAM fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('Starting local FastAPI server on port 8081 (loading models into GPU, please wait 1-3 mins)...')
server_process = subprocess.Popen([
    'python', 'api_server.py',
    '--host', '127.0.0.1',
    '--port', '8081'
])

print('Starting Cloudflare Tunnel...')
tunnel_process = subprocess.Popen([
    'cloudflared', 'tunnel',
    '--url', 'http://127.0.0.1:8081'
], stderr=subprocess.PIPE, universal_newlines=True)

public_url = None
for line in tunnel_process.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print(f'Cloudflare Tunnel active: {public_url}')
print('Waiting for Hunyuan3D model weights to finish loading into GPU VRAM...')

# Healthcheck loop: Wait until FastAPI is listening and /health responds 200 OK
server_ready = False
for i in range(120):
    if server_process.poll() is not None:
        print(f'Error: api_server.py exited prematurely with code {server_process.returncode}!')
        break
    try:
        req = urllib.request.urlopen('http://127.0.0.1:8081/health', timeout=2)
        if req.getcode() == 200:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(3)
    if (i + 1) % 10 == 0:
        elapsed = (i + 1) * 3
        print(f'Still loading model weights into GPU VRAM... ({elapsed}s elapsed)')

if server_ready:
    print('\n' + '='*60)
    print('🎉 HUNYUAN3D CLOUD SERVER IS READY FOR UNITY!')
    print('='*60)
    print(f'👉 Copy this Server URL into Unity: {public_url}')
    print('='*60 + '\n')
else:
    print('\n[!] Server startup timed out or encountered an error. Check server logs.')

# Keep running
server_process.wait()
